# CDOT TDM VMT Calculator

Worked example per strategy on real CDOT TAZs. Nineteen strategies total: sixteen consolidated from the formulas in `Methods_Research_Updated.xlsx`, plus three additional strategies (Park and Ride, Mobility Hub, Traffic Calming) implemented using the alternative approaches Handy et al. 2025 recommends for strategies they flag as 'quantification not recommended'.

**Pipeline**
1. `prepare_taz.prepare_taz()` builds the 8,045-row per-TAZ table from local TDM files + cached CDOT layers + cached background data (ACS mode share, NOAA bikeable days).
2. `add_imputed_mode_shares` / `add_imputed_avo` / `add_imputed_parking` fill the behavioral columns the TDM doesn't carry. **When `data/external/` contains background data (populated by `scripts/fetch_background_data.py`), observed values take precedence per-TAZ**; the area-type defaults are used only as fallback for TAZs not covered.
3. Each `strategy_*` function returns a per-TAZ result row with `pct_vmt_reduction`, `daily_vmt_reduction`, and a `data_assumptions` flag listing which defaults were used. The flag distinguishes `acs_b08301_commute` / `noaa_taz_idw` (real observed data) from `imputed_from_area_type` (fallback).

**Data sources for defaults**

| Default | Source | Granularity | Coverage |
|---|---|---|---|
| Transit / auto / bike / walk mode share | ACS B08301 2022 5-Year | Block group (via `geoid`) | ~70% of CO TAZs |
| Annual bikeable days | NOAA NCEI 1991-2020 **Daily** Climate Normals, IDW-interpolated (k=5, p=2) to TAZ centroids from 30 HCN/CRN/GSN stations | Per-TAZ (via centroid IDW) | **100% of CO TAZs** |
| AVO, parking price, share paying, transit fare | Area-type defaults (NHTS, AAA, agency tariffs) | 4 area-type buckets | 100% (fallback) |

**Bikeable-days definition.** For each calendar day of the year:
- contribution = 0 if `TMAX_NORMAL` is outside [32°F, 95°F] (whole-day freeze or extreme heat)
- contribution = `1 − precip_probability` otherwise

The yearly sum gives the expected number of bikeable days — no inclusion-exclusion needed because each calendar day either satisfies the temperature condition or doesn't, and precipitation probability is folded in multiplicatively.

**Sign convention**
- `pct_vmt_reduction` < 0 → VMT reduction; > 0 → VMT increase (induced demand, fare hike).
- `daily_vmt_reduction` > 0 → miles saved per day; < 0 → miles added per day.

**Stacking warning.** Strategy results are computed independently. To combine, use multiplicative stacking `retained_vmt = product(1 + r_i)` — do NOT add the percentages. Some pairs target the same decision and must not be stacked (e.g., Parking Pricing with `trip_purpose='commute'` + Parking Cash-Out).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from prepare_taz import prepare_taz
import strategy_calculations as sc

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pd.set_option('display.max_columns', 60)

## 1. Load the per-TAZ data and impute missing behavioral inputs

In [ ]:
prep = prepare_taz()
taz = sc.add_imputed_parking(sc.add_imputed_avo(sc.add_imputed_mode_shares(prep.df)))

print(f'{len(taz):,} TAZs x {taz.shape[1]} columns')
print()
print('Area type distribution:')
print(taz['area_type'].value_counts())

## 2. Pick representative TAZs for demos

In [ ]:
demo_taz_ids = (
    taz[taz['area_type'] == 'urban_core'].nlargest(1, 'activity_density')['taz_id'].tolist()
    + taz.nlargest(1, 'employment')['taz_id'].tolist()
    + taz.nlargest(1, 'population')['taz_id'].tolist()
    + taz[taz['area_type'] == 'suburban'].nlargest(1, 'employment')['taz_id'].tolist()
    + taz[taz['area_type'] == 'rural'].nlargest(1, 'daily_vmt')['taz_id'].tolist()
)
demo_taz_ids = list(dict.fromkeys(demo_taz_ids))
demo = taz[taz['taz_id'].isin(demo_taz_ids)].copy()
demo[['taz_id', 'county', 'district', 'area_type', 'population', 'employment',
      'daily_vmt', 'pace_lts_avg', 'transit_stop_count']]

## Transit (2 strategies)

### 1. Transit Service Expansion

Covers Methods rows 1-3 (Increased Frequency, New Intercity Service, New Local Service). `basis='frequency'` (Handy 2013 e=0.50) or `basis='service_miles'` (TCRP 95 e=0.75).

In [ ]:
served = taz[taz['transit_vrh'] > 0].nlargest(5, 'transit_vrh')
freq = sc.strategy_transit_service_expansion(served, pct_change=0.25, basis='frequency',
                                              level_of_implementation=0.60)
miles = sc.strategy_transit_service_expansion(served, pct_change=0.20, basis='service_miles')
pd.concat([freq, miles], ignore_index=True)

### 2. Transit Fare Subsidy

Covers Methods rows 18 and 22 (Pass Subsidy + Employee Commute Benefits / ECO Pass). `scope='all'` (general) or `scope='commute'` (employer-sponsored).

In [ ]:
general = sc.strategy_transit_fare_subsidy(served, pct_fare_reduction=0.50,
                                            pct_eligible=0.40, scope='all')
eco_pass = sc.strategy_transit_fare_subsidy(served, subsidy_amount=2.00,
                                             pct_eligible=0.50, scope='commute')
pd.concat([general, eco_pass], ignore_index=True)

## Bike (3 strategies)

### 3. Separated & Protected Bike Lanes

Methods row 4. Best for TAZs with high LTS (currently bike-hostile).

In [ ]:
bike_candidates = (taz[(taz['pace_segments_mi'] > 1) & (taz['pace_lts_avg'] > 3)]
                   .nlargest(5, 'pace_no_facility_share'))
sc.strategy_separated_bike_lanes(bike_candidates, pct_parallel_vmt_affected=0.05)

### 4. Bike Mode-Share Booster

Covers Methods rows 5 and 23. `scope='area_vmt'` (sharrows, default +15% boost) or `scope='commute'` (end-of-trip facilities, default +5%).

In [ ]:
sharrows = sc.strategy_bike_mode_share_booster(demo, scope_share=0.10, scope='area_vmt')
end_of_trip = sc.strategy_bike_mode_share_booster(demo.nlargest(5,'employment'),
                                                   scope_share=0.40, scope='commute')
pd.concat([sharrows, end_of_trip], ignore_index=True)

### 5. Shared Micromobility

Methods row 6.

In [ ]:
micro_candidates = taz[taz['area_type'] == 'urban_core'].nlargest(5, 'activity_density')
sc.strategy_shared_micromobility(micro_candidates,
                                  pct_pop_access_before=0.0,
                                  pct_pop_access_after=0.30)

## Land Use (3 strategies)

### 6. Density Change

Covers Methods rows 7 and 8. Pass only `pct_change_res_density` for residential-only; pass both for mixed-use.

In [ ]:
res_only = sc.strategy_density_change(demo, pct_change_res_density=0.20)
mixed_use = sc.strategy_density_change(demo, pct_change_res_density=0.20,
                                        pct_change_emp_density=0.20)
pd.concat([res_only, mixed_use], ignore_index=True)

### 7. Transit Oriented Development

Methods row 9. Default TOD transit-share boost = +10%.

In [ ]:
tod_candidates = taz.nlargest(5, 'transit_stop_count')
sc.strategy_transit_oriented_development(tod_candidates)

### 8. Affordable Housing / Infill

Methods row 24.

In [ ]:
sc.strategy_affordable_housing(demo, pct_units_affordable=0.30)

## Parking (3 strategies)

### 9. Parking Pricing

Covers Methods rows 12-14. `trip_purpose='commute'` (workplace) or `trip_purpose='all'` (curb/dynamic).

In [ ]:
emp_centers = taz.nlargest(5, 'employment')
workplace = sc.strategy_parking_pricing(emp_centers, new_price=15.0,
                                         trip_purpose='commute')
curb_mgmt = sc.strategy_parking_pricing(emp_centers, new_price=8.0,
                                         trip_purpose='all', share_affected=0.30,
                                         existing_price=4.0)
pd.concat([workplace, curb_mgmt], ignore_index=True)

### 10. Unbundled Parking (Multifamily)

Methods row 10.

In [ ]:
multifamily = taz[taz['area_type'].isin(['urban_core','urban'])].nlargest(5, 'households')
sc.strategy_unbundled_parking(multifamily, annual_parking_cost=1800)

### 11. Parking Cash-Out

Methods row 11. **Do not stack with Parking Pricing (commute)** — both target the same employee parking decision.

In [ ]:
sc.strategy_parking_cashout(emp_centers, pct_eligible_employees=0.40)

## Vanpool & Commute Programs (4 strategies)

### 12. Vanpool

Methods row 15. Best for long suburban-to-employment commute corridors.

In [ ]:
vanpool_candidates = taz[(taz['area_type'] == 'suburban') & (taz['employment'] > 1000)].nlargest(5, 'employment')
sc.strategy_vanpool(vanpool_candidates, pct_trips_impacted=0.05)

### 13. TMO Coverage

Covers Methods rows 16 and 17 (formula was already identical for new and join).

In [ ]:
sc.strategy_tmo_coverage(emp_centers, share_before=0.0, share_after=0.40)

### 14. Commute Program (Marketing / Incentives)

Covers Methods rows 20 and 21. Override `reduction_per_eligible` for marketing-only (~1%) or incentive-only (~3%).

In [ ]:
marketing = sc.strategy_commute_program(emp_centers, pct_eligible=0.60,
                                         reduction_per_eligible=0.01)
incentives = sc.strategy_commute_program(emp_centers, pct_eligible=0.30,
                                          reduction_per_eligible=0.03)
pd.concat([marketing, incentives], ignore_index=True)

### 15. Telework

Methods row 19.

In [ ]:
sc.strategy_telework(emp_centers, pct_eligible=0.50, telework_days_per_week=2)

## Induced Demand (1 strategy)

### 16. Lane-Mile Addition

Methods row 25. Adding capacity **increases** VMT under Duranton & Turner 2011.

In [ ]:
arterial_candidates = taz[taz['lane_mi_major_arterial'] > 5].nlargest(5, 'lane_mi_major_arterial')
sc.strategy_lane_mile_addition(arterial_candidates, new_lane_miles=2.0,
                                facility_class='major_arterial')

## Additional strategies (Handy 2025 'quantification not recommended')

Three strategies the spreadsheet flagged with Handy et al. 2025's "quantification not recommended" guidance. Each is implemented using Handy's suggested alternative approach.

**Sources for these three**
- Park and Ride: CAPCOA (2010, 2021) Measure T-22; TCRP Synthesis 69 (2007); Litman / VTPI Park & Ride Encyclopedia.
- Mobility Hub: SANDAG (2021) Mobility Hubs Implementation Strategy; Caltrans (2023) Mobility Hubs Practitioner Guide; LADOT (2020) Mobility Hubs Reader's Guide; TCRP Report 188 (2017).
- Traffic Calming: Stevens (2016) intersection-density elasticity (JAPA 83:1); CAPCOA (2021) T-19 and T-20; Ewing et al. (2020); FHWA (2017) Traffic Calming ePrimer; Boarnet et al. (2017).

### 17. Park and Ride (CAPCOA T-22 direct trip substitution)

Implementation: a P&R user replaces home→work auto trip with home→P&R auto trip + P&R→work transit trip. Per-user daily auto VMT reduction = 2 × (commute distance − distance to lot). Capped at 1% of daily VMT (CAPCOA T-22 ceiling).

Below: 200 P&R users resident in suburban TAZs, lots ~3 miles from home.

In [ ]:
# P&R works best in suburban TAZs along intercity transit corridors
pr_candidates = taz[(taz['area_type'] == 'suburban') & (taz['population'] > 500)].nlargest(5, 'daily_vmt')
sc.strategy_park_and_ride(pr_candidates, n_pr_users=200, distance_to_pr_mi=3.0)

### 18. Mobility Hub (composite stack)

Implementation per Handy et al. 2025 "alternative interactions" guidance: stack the constituent strategies (micromobility + transit frequency + end-of-trip facilities) multiplicatively, scale by catchment-area share, apply a CAPCOA-style 0.80 discount to handle overlapping user populations.

Best applied to dense urban_core TAZs with existing transit nodes. Below: a typical urban hub deployment using the default component bundle.

In [ ]:
hub_candidates = taz[(taz['area_type'] == 'urban_core') & (taz['transit_stop_count'] >= 5)].nlargest(5, 'transit_stop_count')
default_hub = sc.strategy_mobility_hub(hub_candidates, catchment_share=0.30)
default_hub

In [ ]:
# Same hub_candidates with an enhanced configuration: 50% catchment, no discount, larger components
enhanced_hub = sc.strategy_mobility_hub(
    hub_candidates,
    catchment_share=0.50,
    composite_discount=1.0,
    components={
        'shared_micromobility':    dict(pct_pop_access_before=0.0, pct_pop_access_after=0.50),
        'transit_frequency_boost': dict(pct_change=0.20, basis='frequency',
                                         level_of_implementation=1.0),
        'end_of_trip_facilities':  dict(scope_share=0.60, scope='commute'),
    },
)
enhanced_hub

### 19. Traffic Calming (basis switch)

Two parallel formulations:
- `basis='connectivity'` (Handy 2025's preferred alternative): uses Stevens 2016 intersection-density elasticity (-0.12). Treats calming as a connectivity improvement.
- `basis='mode_shift'` (CAPCOA T-19/T-20 path): boosts bike and walk mode share on the calmed network share.

Caveat: do NOT stack the `mode_shift` form with Bike Mode-Share Booster on the same TAZ (risks double-counting bike facility effect).

In [ ]:
# Best applied to dense TAZs currently bike-hostile (high LTS)
calming_candidates = taz[(taz['area_type'].isin(['urban','urban_core'])) & (taz['pace_lts_avg'] >= 3)].nlargest(5, 'pop_density')
connectivity_form = sc.strategy_traffic_calming(calming_candidates, pct_calmed_share=0.20, basis='connectivity')
mode_shift_form = sc.strategy_traffic_calming(calming_candidates, pct_calmed_share=0.20, basis='mode_shift')
pd.concat([connectivity_form, mode_shift_form], ignore_index=True)

## Stacking multiple strategies on the same TAZ

Apply an employer-focused TDM bundle and combine reductions multiplicatively.

**Stacking rules**
- `retained_vmt = product(1 + r_i)` where `r_i` is each strategy's `pct_vmt_reduction`.
- **Do not stack** Parking Pricing (commute) + Parking Cash-Out — both target the same employee decision.
- The bundle below is safe: TMO + telework + ECO Pass + workplace parking pricing act through partially distinct mechanisms.

In [ ]:
bundle_taz = taz.nlargest(3, 'employment')

results = [
    sc.strategy_tmo_coverage(bundle_taz, share_before=0.0, share_after=0.40),
    sc.strategy_parking_pricing(bundle_taz, new_price=15.0, trip_purpose='commute'),
    sc.strategy_transit_fare_subsidy(bundle_taz, subsidy_amount=2.00,
                                      pct_eligible=0.50, scope='commute'),
    sc.strategy_telework(bundle_taz, pct_eligible=0.30, telework_days_per_week=2),
]

long = pd.concat(results, ignore_index=True)
long[['taz_id','strategy','pct_vmt_reduction','daily_vmt_reduction','data_assumptions']]

In [ ]:
def stack(group):
    combined_pct = (1.0 + group['pct_vmt_reduction']).prod() - 1.0
    base = group['base_vmt'].max()
    return pd.Series({
        'strategies_applied': ', '.join(group['strategy'].tolist()),
        'combined_pct_reduction': combined_pct,
        'combined_daily_vmt_reduction': -base * combined_pct,
    })

stacked = long.groupby('taz_id').apply(stack, include_groups=False).reset_index()
stacked.merge(bundle_taz[['taz_id','county','employment','daily_vmt']], on='taz_id')

## Data-assumption audit

Run every strategy on the demo TAZs and flag which ones used imputed defaults.

In [ ]:
audit_runs = [
    ('transit_service_expansion',     dict(pct_change=0.25, basis='frequency',
                                            level_of_implementation=0.60)),
    ('transit_fare_subsidy',          dict(pct_fare_reduction=0.50, pct_eligible=0.40)),
    ('separated_bike_lanes',          dict(pct_parallel_vmt_affected=0.05)),
    ('bike_mode_share_booster',       dict(scope_share=0.10, scope='area_vmt')),
    ('shared_micromobility',          dict(pct_pop_access_before=0.0,
                                            pct_pop_access_after=0.30)),
    ('density_change',                dict(pct_change_res_density=0.20)),
    ('transit_oriented_development',  dict()),
    ('affordable_housing',            dict(pct_units_affordable=0.30)),
    ('parking_pricing',               dict(new_price=15.0, trip_purpose='commute')),
    ('unbundled_parking',             dict(annual_parking_cost=1800)),
    ('parking_cashout',               dict(pct_eligible_employees=0.40)),
    ('vanpool',                       dict(pct_trips_impacted=0.05)),
    ('tmo_coverage',                  dict(share_before=0.0, share_after=0.40)),
    ('commute_program',               dict(pct_eligible=0.50)),
    ('telework',                      dict(pct_eligible=0.50, telework_days_per_week=2)),
    ('lane_mile_addition',            dict(new_lane_miles=2.0, facility_class='major_arterial')),
    ('park_and_ride',                 dict(n_pr_users=200, distance_to_pr_mi=3.0)),
    ('mobility_hub',                  dict(catchment_share=0.30)),
    ('traffic_calming',               dict(pct_calmed_share=0.20, basis='connectivity')),
]

audit_rows = []
for name, kw in audit_runs:
    r = sc.STRATEGY_REGISTRY[name](demo, **kw)
    flagged = (r['data_assumptions'].astype(str).str.len() > 0).sum()
    audit_rows.append({
        'strategy': r['strategy'].iloc[0],
        'rows_with_imputed_defaults': flagged,
        'sample_assumption': r['data_assumptions'].iloc[0] or '(none - observed TAZ data only)',
    })
pd.DataFrame(audit_rows)